# Week 10 prep: averaging the latest polls

This notebook turns two big files of polls, `house.csv` and `senate.csv`, into two small, clean files that are ready to map:

* `house_average_recent.csv`: one row for each U.S. House district
* `senate_average_recent.csv`: one row for each state with a Senate race

The mapping notebook (`week10_scratch.ipynb`) only reads those two files. Everything here is **data wrangling** with pandas, and nothing is drawn.

**The plan**

1. Load the polls and keep the district-level (or state-level), general-election rows.
2. Reshape so each poll is one row, with a Democratic number and a Republican number.
3. For each place, average its latest week of polling.
4. Save the result as a CSV.

Both files have the same columns, so we'll write the steps as functions and use them twice.

In [ ]:
import pandas as pd

DAYS = 7           # how many days of polling to average, counting back from each place's latest poll

## 1. Look at the polls

Each row is **one answer to one poll question**, so a poll between two candidates takes two rows.

In [ ]:
house = pd.read_csv("house.csv")
senate = pd.read_csv("senate.csv")

# dates look like 10/30/26, so we tell pandas the format
house["end_date"] = pd.to_datetime(house["end_date"], format="%m/%d/%y")
senate["end_date"] = pd.to_datetime(senate["end_date"], format="%m/%d/%y")

print("house:", house.shape, " senate:", senate.shape)
house[["poll_id", "pollster", "state", "seat_number", "end_date", "stage", "party", "pct"]].head(6)

A House race is a **district** (a state plus a `seat_number`). A Senate race is a whole **state**, so `seat_number` is empty. We'll call these the **places** of each file: `["state", "seat_number"]` for the House, and `["state"]` for the Senate.

## 2. Filter and reshape

Rows where `state` is `US` are *national* polls, which we don't want. We want:

* polls of a place (`state` isn't `US`),
* for the November election (`stage` is `general`) in 2026,
* and just the Democratic and Republican candidates.

Then `pivot_table()` turns each poll's two party rows into two columns, `DEM` and `REP`, and the **margin** is the Democratic percentage minus the Republican percentage. A positive margin is a Democratic lead.

Some polls list several candidates from one party (for example, a Republican and a write-in). `aggfunc="max"` keeps the **highest** number for each party, which is the main candidate.

In [ ]:
def one_row_per_poll(polls, places):
    """Keep the general-election polls, with one row per poll and a DEM, REP and margin column."""
    general = polls[(polls["state"] != "US") & (polls["stage"] == "general") & (polls["cycle"] == 2026)]
    two_party = general[general["party"].isin(["DEM", "REP"])]

    by_poll = two_party.pivot_table(
        index=["poll_id", "question_id", "end_date"] + places,
        columns="party",
        values="pct",
        aggfunc="max",
    ).reset_index()

    by_poll = by_poll.dropna(subset=["DEM", "REP"])      # keep the polls that have both numbers
    by_poll["margin"] = by_poll["DEM"] - by_poll["REP"]
    return by_poll


house_places = ["state", "seat_number"]
senate_places = ["state"]

house_polls = one_row_per_poll(house, house_places)
senate_polls = one_row_per_poll(senate, senate_places)

print(len(house_polls), "House polls,", len(senate_polls), "Senate polls")
house_polls.head()

## 3. Average the latest week

What is "the latest week"? The simplest answer is the last 7 days of the whole dataset. Let's see how that works:

In [ ]:
for name, by_poll, places in [("House", house_polls, house_places), ("Senate", senate_polls, senate_places)]:
    latest_poll = by_poll["end_date"].max()
    last_week = by_poll[by_poll["end_date"] > latest_poll - pd.Timedelta(days=7)]

    print(name, "- most recent poll ends", latest_poll.date())
    print("   places with a poll in the last 7 days:", last_week.groupby(places).ngroups)
    print("   places with any polling at all:       ", by_poll.groupby(places).ngroups)

Few places are polled in any given week, so the map would be nearly empty. Instead, for **each place**, we find its own most recent poll and average the polls within 7 days of it (`DAYS`).

Because a place's "latest" poll can be old, we also save when it was (`latest`) and how many days before the newest poll in the data that was (`days_old`), so nobody is misled by an old number.

In [ ]:
def average_latest_week(by_poll, places, days):
    """Average each place's polls from the `days` before its own most recent poll."""
    by_poll = by_poll.copy()
    by_poll["place_latest"] = by_poll.groupby(places)["end_date"].transform("max")
    recent = by_poll[by_poll["end_date"] > by_poll["place_latest"] - pd.Timedelta(days=days)]

    averages = recent.groupby(places).agg(
        dem=("DEM", "mean"),
        rep=("REP", "mean"),
        polls=("poll_id", "nunique"),
        latest=("end_date", "max"),
    ).reset_index()

    averages["margin"] = averages["dem"] - averages["rep"]
    averages["days_old"] = (by_poll["end_date"].max() - averages["latest"]).dt.days
    averages["latest"] = averages["latest"].dt.strftime("%Y-%m-%d")
    return averages.round(1)

### The House

In [ ]:
house_average = average_latest_week(house_polls, house_places, DAYS)

house_average["district"] = house_average["seat_number"].astype(int)
house_average["id"] = house_average["state"] + "-" + house_average["district"].astype(str)      # like "NY-17"

house_average = house_average[["id", "state", "district", "dem", "rep", "margin", "polls", "latest", "days_old"]]

print(len(house_average), "districts")
house_average.sort_values("margin").head()

### The Senate

In [ ]:
senate_average = average_latest_week(senate_polls, senate_places, DAYS)

senate_average = senate_average[["state", "dem", "rep", "margin", "polls", "latest", "days_old"]]

print(len(senate_average), "states")
senate_average.sort_values("margin").head()

## 4. Check, then save

Before saving, look for problems: are there districts with only one poll? Is anything very old?

In [ ]:
for name, averages in [("House", house_average), ("Senate", senate_average)]:
    print(name)
    print("   Democratic lead:", (averages["margin"] > 0).sum(), "  Republican lead:", (averages["margin"] < 0).sum(), "  tied:", (averages["margin"] == 0).sum())
    print("   based on a single poll:", (averages["polls"] == 1).sum(), "  latest poll over 30 days old:", (averages["days_old"] > 30).sum())

In [ ]:
house_average.to_csv("house_average_recent.csv", index=False)
senate_average.to_csv("senate_average_recent.csv", index=False)

pd.read_csv("senate_average_recent.csv").head()

Open the saved files in a text editor to see how small and plain they are. These are the files the mapping notebook uses.

**Things to try**

* Change `DAYS` to 14 or 30, rerun the notebook, and see how the averages and the number of polls change.
* Look at `house_average.sort_values("margin").head(3)`. In AK-1, the Democrat has only about 6%, because the main challenger is an independent. Our DEM-versus-REP margin misses that. How would you handle races like this?
* Which places are the most uncertain? Sort by `polls`, or by `days_old`.
* Are there other ways to average? For example, weighting polls with a larger `sample_size` more heavily.